In [420]:
import numpy as np
import matplotlib.pyplot as plt

## **Exercise 2.1.1**

1. **Write a function** that accepts an upper triangular matrix and implements the backward substitution. Test is on the linear system $U\mathbf{x} = \mathbf{b}$ with
   $$U = \begin{pmatrix} 2 & 1 & 1 \\ 0 & 1 & -2 \\ 0 & 0 & 1 \end{pmatrix}, \quad \mathbf{b} = \begin{pmatrix} 1 \\ -1 \\ 4 \end{pmatrix} \tag{2.1.10}$$
   and check that the solution is correct.

   > **Note:** You will use this function later on, so try to keep the code as modular as possible!

2. **Write a separate function** that performs Gaussian elimination and combine it with backward substitution to solve the system
   $$\begin{pmatrix} 2 & 1 & 1 \\ 1 & 1 & -2 \\ 1 & 2 & 1 \end{pmatrix} \begin{pmatrix} x_0 \\ x_1 \\ x_2 \end{pmatrix} = \begin{pmatrix} 8 \\ -2 \\ 2 \end{pmatrix} \tag{2.1.11}$$
   Check the solution. See the Appendices *Matrices* and *Modularity* for suggestions.

3. **Using the previous programs**, add a function that prints also the explicit inverse of a matrix. Test it on the matrix above and check the result.

4. **Adapt the previous programs** to include the partial pivoting and solve the following systems. Check that without partial pivoting the program fails, since at some intermediate step $A_{jj} = 0$ appears for the matrix
   $$\begin{pmatrix} 2 & 1 & 1 \\ 2 & 1 & -4 \\ 1 & 2 & 1 \end{pmatrix} \begin{pmatrix} x_0 \\ x_1 \\ x_2 \end{pmatrix} = \begin{pmatrix} 8 \\ -2 \\ 2 \end{pmatrix} \tag{2.1.12}$$

5. **Study the solution** to the problem
   $$\begin{pmatrix} 10^{-20} & -1 \\ 1 & 1 \end{pmatrix} \begin{pmatrix} x_0 \\ x_1 \end{pmatrix} = \begin{pmatrix} 1 \\ 2 \end{pmatrix} \tag{2.1.13}$$
   with and without pivoting. In this case the system is not ill-conditioned, i.e. it does not generate infinities, but produces the wrong result. Explain the origin of the error and why pivoting solves it.

## ***Backward Substitution* algorithm**

In [421]:
def forw_subst (L_in, b_in, speak=False):
    U = np.copy(L_in)
    b = np.copy(b_in)
    if np.shape(U)[0] != np.shape(U)[1]:
        raise ValueError("Must use a square matrix!")

    n = np.shape(U)[0]
    np_xi = np.zeros(n)

    for i in range(n):
        x_i = (b[i] - np.dot(U[i, :i], np_xi[:i])) / U[i, i]
        np_xi[i] = x_i

    test = np.allclose(np.dot(U, np_xi), b) 
    if speak:
        print('\nSolution found:\nx =', np_xi)
        print(f'\nTest true answer: {test}')
        # print('b =      ', b)
        # print('U • x =  ', np.dot(U, np_xi))
    
    return np_xi

def back_subst (U_in, b_in, speak=False):
    U = np.copy(U_in)
    b = np.copy(b_in)
    if np.shape(U)[0] != np.shape(U)[1]:
        raise ValueError("Must use a square matrix!")

    n = np.shape(U)[0]
    np_xi = np.zeros(n)
    
    for i in reversed(range(n)):
        x_i = (b[i] - np.dot(U[i, i+1:], np_xi[i+1:])) / U[i, i]
        np_xi[i] = x_i

    test = np.allclose(np.dot(U, np_xi), b) 
    if speak:
        print('\nSolution found:\nx =', np_xi)
        print(f'\nTest true answer: {test}')    # Prints True if the answer is correct after the check with b
        # print('b =      ', b)
        # print('U • x =  ', np.dot(U, np_xi))
                
    return np_xi

In [422]:
test_A = np.array([
    [2, 1, 1],
    [0, 1, -2],
    [0, 0, 1]], dtype=float)
test_b = np.array([1, -1, 4], dtype=float)

test_TA = test_A.T
test_Tb = np.flip(test_b)

print('Algorithms start!')

x = back_subst (test_A, test_b, speak=True)
print()
y = forw_subst (test_TA, test_Tb, speak=True)

Algorithms start!

Solution found:
x = [-5.  7.  4.]

Test true answer: True


Solution found:
x = [ 2. -3. -7.]

Test true answer: True


## ***Gaussian Elimination* algorithm**

In [423]:
def BackGauss_dumb (A_in, b_in): 
    A = np.copy(A_in)
    b = np.copy(b_in)
    n = np.shape(A)[0]
    for j in range(n-1):
        for i in range(j+1, n):
            c = A[i,j] / A[j,j]
            A[i,:] -= c * A[j,:]
            b[i] -= c * b[j]
    return back_subst (A, b)

In [424]:
test_A = np.array([
    [2, 1,  1],
    [1, 1, -2],
    [1, 2,  1]], dtype=float)

test_b = np.array([8, -2, 2], dtype=float)

xi = BackGauss_dumb (test_A, test_b)
test = np.allclose(test_A @ xi, test_b)

print(test_A, '\n')
print('Algorithm starts!\n')
print('x =', xi)

print(f'\nTest true answer: {test}')    # Prints True if the answer is correct after the check with b


[[ 2.  1.  1.]
 [ 1.  1. -2.]
 [ 1.  2.  1.]] 

Algorithm starts!

x = [ 4. -2.  2.]

Test true answer: True


## ***Inverse of a Matrix* algorithm**

In [425]:
def matrix_inverse(A, speak=False):
# Computes inverse of a matrix
    n = A.shape[0]
    A_inv = np.zeros_like(A, dtype=float)
    
    # Cycle on each of Id mat cols
    for i in range(n):
        e = np.zeros(n)
        e[i] = 1  # i-col of Id mat
        # solve A x = e using gaussian elimination + backward
        x = BackGauss_dumb (A.copy(), e)
        # solution becomes col of inverse
        A_inv[:, i] = x
    
    if speak:
        test = np.allclose(A @ A_inv, np.eye(A.shape[0]))
        print('Solution found:\nA^-1 =\n', A_inv)
        print(f'\nTest true answer: {test}')    # Prints True if the answer is correct
    return A_inv

In [426]:
test_A = np.array([
    [2, 1,  1],
    [1, 1, -2],
    [1, 2,  1]], dtype=float)

print(test_A, '\n')
x = matrix_inverse(test_A, True)

[[ 2.  1.  1.]
 [ 1.  1. -2.]
 [ 1.  2.  1.]] 

Solution found:
A^-1 =
 [[ 0.625  0.125 -0.375]
 [-0.375  0.125  0.625]
 [ 0.125 -0.375  0.125]]

Test true answer: True


## ***Partial Pivoting modifier* algorithm**

In [427]:

test_A = np.array([[2, 1, 1],
                   [2, 1, -4],
                   [1, 2, 1]], dtype=float)

test_b = np.array([8, -2, 2], dtype=float)

try:
    x = BackGauss_dumb (test_A, test_b)
    print("Solution:", x)
except Exception as e:
    print("An error occured!!:", e)

Solution: [nan nan nan]


C:\Users\Marco\AppData\Local\Temp\ipykernel_14020\2611833494.py:7: RuntimeWarning: divide by zero encountered in scalar divide
  c = A[i,j] / A[j,j]
C:\Users\Marco\AppData\Local\Temp\ipykernel_14020\2611833494.py:8: RuntimeWarning: invalid value encountered in multiply
  A[i,:] -= c * A[j,:]
C:\Users\Marco\AppData\Local\Temp\ipykernel_14020\871988014.py:33: RuntimeWarning: invalid value encountered in scalar divide
  x_i = (b[i] - np.dot(U[i, i+1:], np_xi[i+1:])) / U[i, i]


In [428]:
def BackGauss(A_in, b_in, speak=False):
    A = np.array(A_in, dtype=float)
    b = np.array(b_in, dtype=float)
    n = np.shape(A)[0]

    for j in range(n-1):
        # Find row with maximum absolute value in column j
        max_row = j + np.argmax(np.abs(A[j:, j]))
        if max_row != j:
            A[[j, max_row]] = A[[max_row, j]]
            b[[j, max_row]] = b[[max_row, j]]
            # print(f"Pivoting: swapped row {j} with row {max_row}")
        
        # Gaussian elimination
        for i in range(j+1, n):
            c = A[i,j] / A[j,j]
            A[i,:] -= c * A[j,:]
            b[i] -= c * b[j]
            
    # Backward substitution
    return back_subst(A, b, speak)


In [429]:
try:
    x = BackGauss(test_A, test_b)
    print("Solution:", x)
except Exception as e:
    print("An error occured!!:", e)

Solution: [ 4. -2.  2.]


## **An example of particular behaviour**

In [430]:
test_A = np.array([
    [1e-18, -1],
    [1, 1]
])

# Definizione del vettore b
test_b = np.array([1, 2])


xi = BackGauss_dumb(test_A, test_b)
test = np.allclose(test_A @ xi, test_b)

print(test_A, '\n')
print('Algorithm starts!\n')
print('x_dumb =', xi)

print(f'\nTest true answer: {test}')    # Prints True if the answer is correct after the check with b

print('result = ', test_A @ xi)
print('true =   ', test_b)


xi = BackGauss(test_A, test_b)
test = np.allclose(test_A @ xi, test_b)

print('\n\nx_true =', xi)

print(f'\nTest true answer: {test}')    # Prints True if the answer is correct after the check with b

print('result = ', test_A @ xi)
print('true =   ', test_b)

[[ 1.e-18 -1.e+00]
 [ 1.e+00  1.e+00]] 

Algorithm starts!

x_dumb = [ 0. -1.]

Test true answer: False
result =  [ 1. -1.]
true =    [1 2]


x_true = [ 3. -1.]

Test true answer: True
result =  [1. 2.]
true =    [1 2]
